## Assignment V: Determinants, Decompositions, and Eigenspaces

**Due: Friday, December 4, 11:59 PM.**

> **This assignment is optional.**
> - If you **do not submit** it, you are exempted: it will not count toward your grade.
> - If you **submit** it, the grade you receive **will be counted**, whatever that grade is.
>
> Take a look and decide whether it is worth it for you; it is meant as an extra chance to practice the material.

**Total: 50 points.** Questions 1 and 2 are 25 points each.

This assignment uses the class notebooks on **determinants**, **matrix decompositions** (LU, QR, SVD) and **eigenspaces**.

- Each question is split into **Part (a)** and **Part (b)**.
- Write your code in the cell provided for each part. Make sure it **prints** (or `display`s) your answers clearly, ideally with an f-string or a label that says what the printed value represents.
- Use **SymPy** for exact answers and **NumPy/SciPy** to check them, as in the class notebooks. Use `@` for matrix products.
- Some parts ask a written question. Answer those in the **markdown cell** provided right below the code cell.
- **Before you submit, make sure that all outputs are visible in the notebook.** Run every cell; answers that do not appear in the submitted notebook cannot be graded.
- **Submission:** see the instructions at the very end of this notebook.

### AI Usage Policy

**No AI tools (ChatGPT, Claude, GitHub Copilot's suggestion/chat features, etc.) may be used to generate, complete, or rewrite your solutions for this assignment.** All code and explanations must be your own work.

**Exception and disclosure requirement:** If you have **GitHub Copilot active** in your editor (e.g. inline autocomplete suggestions appearing as you type), you do not need to disable it, but you **must disclose this** by adding a short **markdown cell right below this one** stating:
- That Copilot (or any other AI tool) was active while you worked, and
- Briefly, how you used or did not use its suggestions (e.g. "Copilot was active but I did not accept any of its suggestions for graded cells" or "I accepted a Copilot suggestion for the `ax.set_xlabel` lines in Q2(a) but wrote the rest myself").

Notebooks with AI-generated solutions that are not disclosed as above will not receive credit, regardless of correctness.

**AI Usage Disclosure (student to fill in):**

*(If GitHub Copilot or any other AI tool was active while you worked on this assignment, describe that here. If no AI tool was active at all, write "No AI tools were used.")*

Run this cell first (do not change it).

In [ ]:
import sympy as sp
import numpy as np
import scipy.linalg as sla
from IPython.display import display

### Question 1: Planning a Smoothie (25 points)

You make a smoothie from three ingredients: **oats**, **yogurt** and **peanut butter**. Per serving, each ingredient gives the following amounts of protein, carbohydrates and fat (in units of 5 g):

| | oats | yogurt | peanut butter |
|---|---|---|---|
| **protein** | 2 | 1 | 3 |
| **carbs** | 1 | 2 | 1 |
| **fat** | 2 | 4 | $k$ |

Collect these numbers in the $3\times 3$ **nutrition matrix** $N$ (rows = nutrients, columns = ingredients). If you use $x_1, x_2, x_3$ servings of the three ingredients, the nutrients you get are $N\mathbf{x}$.

If $\det N \neq 0$, $N$ is invertible, so **any** target amount of protein, carbs and fat can be reached by some choice of $\mathbf{x}$. If $\det N = 0$, some targets are impossible.

The fat content $k$ of the peanut butter depends on the brand. Run the cell below to define $N$ with a symbol $k$ (do not change it).

In [ ]:
k = sp.symbols("k")
N = sp.Matrix([[2, 1, 3],      # protein
               [1, 2, 1],      # carbs
               [2, 4, k]])     # fat
display(N)

**(a) [12 pts] Determinant with a parameter.**
1. Is $N$ symmetric? Check with SymPy.
2. Compute $\det N$ as a function of $k$ and **factor** it.
3. **Solve** $\det N = 0$. For which brand (value of $k$) can you **not** reach every nutrient target?
4. Substitute that value into $N$ and confirm $\det = 0$. Then check the same with NumPy, using a tolerance `abs(det) < 1e-10`.

In the markdown cell below: for the value of $k$ you found, compare the **carbs** row and the **fat** row of $N$. What do you notice, and why does it make some nutrient targets impossible?

In [ ]:
# TODO: your code for Q1(a) here

**Your answer for Q1(a):**

*(What do you notice about the carbs and fat rows? Why are some targets impossible?)*

**(b) [13 pts] LU decomposition for the brand you actually buy.**
Your peanut butter has $k = 8$. Let $N_8$ be $N$ with $k = 8$ substituted.
1. Compute the LU decomposition of $N_8$ in **SymPy**. Display $L$ and $U$, print the row swaps, and check $LU = N_8$.
2. Compute the LU decomposition in **SciPy** (`sla.lu`) and check $PLU = N_8$ with `np.allclose`.
3. Find $\det N_8$ from the diagonal of $U$ in **both** (for SciPy, $\det P = \pm 1$ fixes the sign; use `np.linalg.det(P)`). Confirm with `N8.det()`.

In the markdown cell below: SymPy's and SciPy's $U$ are different. Why? Are both correct?

In [ ]:
# TODO: your code for Q1(b) here

**Your answer for Q1(b):**

*(Why do SymPy and SciPy give different $U$? Are both correct?)*

### Question 2: Bike Sharing and Streaming Habits (25 points)

**(a) [13 pts] Where do the bikes end up? (eigenvalues and eigenspaces)**

A city bike-sharing system has three stations: **Downtown**, **University** and **Riverside**. Every day, each bike is either returned to the station it left from or taken to another one. The fractions are collected in the **transition matrix**
$$
T = \begin{pmatrix} 0.5 & 0.1 & 0.1 \\ 0.1 & 0.8 & 0.1 \\ 0.4 & 0.1 & 0.8 \end{pmatrix},
$$
where column $j$ says where the bikes that start at station $j$ end up (for example, $40\%$ of the Downtown bikes end the day at Riverside). Each column sums to $1$. If $\mathbf{x}$ lists the numbers of bikes at the three stations today, then tomorrow they are $T\mathbf{x}$.

A distribution that does **not change** from day to day satisfies $T\mathbf{x} = \mathbf{x}$: it is an **eigenvector** with eigenvalue $\lambda = 1$.

1. Build $T$ in **SymPy** with exact fractions (`sp.Rational(1, 10)` etc.).
2. Use `eigenvects()` to find each eigenvalue, its **algebraic** and **geometric** multiplicities, and a basis of each eigenspace $E_\lambda$.
3. For the eigenvector $\mathbf{v}$ with $\lambda = 1$, check that $T\mathbf{v} = \mathbf{v}$.
4. Check with **NumPy**: compute `w, V = np.linalg.eig(T)` and verify $TV = V\operatorname{diag}(w)$.

In [ ]:
# TODO: your code for Q2(a) here

**Your answer for Q2(a):**

*(Which station ends up with the most bikes in the long run? Is any eigenvalue defective, i.e. geometric < algebraic multiplicity?)*

**(b) [12 pts] Streaming hours (SVD).**

A survey records the weekly hours four households spend on two streaming services, plus a **total** column:

| household | service 1 | service 2 | total |
|---|---|---|---|
| A | 4 | 2 | 6 |
| B | 1 | 3 | 4 |
| C | 2 | 0 | 2 |
| D | 5 | 5 | 10 |

Let $H$ be the $4\times 3$ matrix of this table (numbers only).
1. Build $H$ in **NumPy**. Compute its singular values.
2. Compute the **full** SVD and the **reduced** SVD. Print the shapes of $U$, $\Sigma$, $V^T$ in both cases, rebuild the $4\times 3$ matrix $\Sigma$ for the full SVD, and check $U\Sigma V^T = H$ in both cases.

In [ ]:
H = np.array([[4, 2, 6],
              [1, 3, 4],
              [2, 0, 2],
              [5, 5, 10]], dtype=float)

# TODO: your code for Q2(b) here

### Submission Instructions

**Submit this assignment on Canvas only.** There is no GitHub submission for this assignment.

**Due: Friday, December 4, 11:59 PM.** Remember: this assignment is **optional**. If you do not submit it, it does not count. If you submit it, your grade on it **is counted**.

1. Before submitting, choose **Kernel → Restart & Run All** (or **Restart Kernel and Run All Cells**) and check that every cell runs without errors and that **all outputs are visible** in the notebook.
2. Make sure your written answers are filled in, in the markdown cells marked *Your answer*, and that the AI Usage Disclosure at the top is filled in.
3. Upload the completed `.ipynb` file to the Assignment V page on Canvas.

No late submissions will be accepted.